In [143]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import factorial, eval_hermite
from numpy.polynomial.hermite import hermgauss
from qutip import (basis, squeeze, tensor, sigmax, sigmay, sigmaz,
                   identity, destroy, displace, position, qeye)


In [144]:
def get_lchs_states(beta, r_target, r_prime, N_dim, n_quad_points=100):
    """
    Calculates the LCHS coefficients C_n using Gauss-Hermite quadrature.
    Corrects for the normalization factor via explicit state normalization.
    """
    print(f"1. Preparing States (N={N_dim}, r_target={r_target})...")
    # 1. Parameters
    gamma = np.exp(-2 * r_prime) - np.exp(-2 * r_target)
    width_param = np.exp(r_prime)
    scale_factor = np.sqrt(2) * width_param

    # 2. Quadrature Setup (Integration over x)
    x_roots, weights = hermgauss(n_quad_points)
    # Map roots to physical variable p (or x in the integral)
    p_points = x_roots * scale_factor

    # 3. Target Function Evaluation g(x)
    # The LCHS target function for the wave equation/CHO
    C_val = 2 * np.pi * np.exp(-2 * beta)
    term_exp = np.exp((1 + 1j * p_points)**beta)
    denom_func = C_val * term_exp * (1 - 1j * p_points)

    # We integrate Target(p) * Basis(p).
    # The weight w_i implicitly includes e^{-x^2}, which cancels the basis Gaussian.
    # We only include the 'excess' Gaussian decay e^{-gamma x^2} from the target.
    target_part = np.exp(-gamma * p_points**2) / denom_func

    # 4. Expansion Coefficients C_n
    cn_list = []
    sqrt_pi = np.sqrt(np.pi)
    basis_prefactor = 1.0 / np.sqrt(sqrt_pi * width_param)

    for n in range(N_dim):
        fock_norm = 1.0 / np.sqrt((2**n) * factorial(n))
        H_val = eval_hermite(n, p_points / width_param)

        # Integral: Sum weighted points * Jacobian
        integrand = target_part * (basis_prefactor * fock_norm * H_val)
        val = np.sum(weights * integrand) * scale_factor
        cn_list.append(val)

    # 5. Construct QuTiP Objects
    cn_array = np.array(cn_list)
    psi_seed = sum([cn_array[n] * basis(N_dim, n) for n in range(N_dim)]).unit()

    # Apply Squeezing (Basis Transform)
    S_op_prime = squeeze(N_dim, r_prime)
    psi_osc_init = S_op_prime * psi_seed

    # displacement = 1j * np.sqrt(2) * beta * np.exp(r_prime)
    # D_op = displace(N_dim, displacement)
    # psi_osc_init = D_op * psi_osc_init

    # Post-Selection State (Target Squeezed State)
    S_op_target = squeeze(N_dim, r_target)
    phi_post = S_op_target * basis(N_dim, 0)

    # phi_post = D_op * phi_post

    print("   States ready.")
    return psi_osc_init, phi_post

In [145]:
def run_simulation(omega, gamma_damp, T_step, n_steps, N_dim, psi_osc_init, phi_post):
    """
    Runs the simulation and extracts both Position (|0>) and Velocity (|1>) components.
    """
    print("2. Running Simulation...")

    # --- Hamiltonian Setup ---
    # These coefficients couple |0> <-> |1> (Pos <-> Vel)
    # Standard mapping for wave eq: du/dt = w*v, dv/dt = -w*u

    L_coeff = 0.5 * (omega**2 - 1)  # Coefficient for Position operator
    H_coeff = -0.5 * (omega**2 + 1) # Coefficient for Momentum operator

    # Damping term: diag(0, 2*gamma_damp) = gamma_damp*(I - Z)
    L_damp = gamma_damp * (qeye(2) - sigmaz())

    Op_L = L_coeff * sigmax() + L_damp
    Op_H = H_coeff * sigmay()

    # Shift L to be PSD (sigma_x has eigenvalues ±1)
    Op_L_psd = Op_L

    a = destroy(N_dim)
    # computational momentum/position operator for the auxiliary space
    K_op = (a.dag() + a) / 2

    kappa = (phi_post.dag() * K_op * psi_osc_init)
    print(kappa)

    # K_op = K_op / np.linalg.norm(kappa)

    # kappa = (phi_post.dag() * K_op * psi_osc_init)
    # print(kappa)

    A_eff = 1j * (kappa * Op_L_psd + Op_H)
    eigs = A_eff.eigenenergies()
    print(eigs)
    omega_eff = float(np.max(np.abs(np.imag(eigs))))
    print(f"   omega_eff={omega_eff:.3f}")

    Id_osc = identity(N_dim)
    Ham_Joint = tensor(K_op, Op_L_psd)
    Ham_Single = tensor(Id_osc, Op_H)

    # Trotter Evolution Step
    U_joint = (-1j * T_step * Ham_Joint).expm()
    U_single = (-1j * T_step * Ham_Single).expm()
    U_step = U_joint * U_single

    # Oscillator-only unitary from PSD shift (commutes with total evolution)
    phase_step = (-1j * T_step * L_shift * K_op).expm()

    # --- Initial State ---
    # Start qubit in |0> (Position = 1, Velocity = 0)
    psi_qubit_0 = basis(2, 0)
    psi_current = tensor(psi_osc_init, psi_qubit_0)

    # Projector to extract qubit state from the joint system
    # We project the oscillator onto <phi_post|
    bra_t = phi_post.dag()
    projector_osc_bra = tensor(bra_t, identity(2))

    # --- Data Collection ---
    u_vals = [] # Position (Real part of |0>)
    v_vals = [] # Velocity (Imag part of |1> or similar depending on encoding)
    times = np.linspace(0, n_steps * T_step, n_steps)

    # Calculate Normalization Factor from t=0
    vec_0 = (projector_osc_bra * psi_current).full()
    norm_factor = np.linalg.norm(vec_0)
    print(f"   Normalization Factor: {norm_factor:.4f}")

    for _ in range(n_steps):
        # 1. Project out the oscillator part -> get 2x1 qubit vector
        projector_osc_bra = tensor(bra_t, identity(2))
        qubit_vec = (projector_osc_bra * psi_current).full()

        # 2. Extract Physical Variables
        # |0> Amplitude -> Position u(t)
        # |1> Amplitude -> Velocity v(t) (rescaled by omega usually, or direct)
        u_t = np.real(qubit_vec[0][0]) / norm_factor

        # Note: Depending on the specific H embedding (sigmax vs sigmay),
        # v(t) might be in the Real or Imaginary part of |1>.
        # For H ~ sigma_y, evolution is Rotation. |1> is -Real or Imag.
        # We plot the magnitude/envelope or Real part to verify oscillation.
        v_t = np.real(qubit_vec[1][0]) / norm_factor

        u_vals.append(u_t)
        v_vals.append(v_t)

        # 3. Evolve
        psi_current = U_step * psi_current
        bra_t = bra_t * phase_step.dag()

    return times, u_vals, v_vals, omega_eff

In [146]:
def main():
    # Parameters
    omega = 3.0
    gamma_damp = 0.1  # damping rate (set < omega for underdamped)
    beta = 0.8
    r_target = 8
    r_prime = 6
    N_dim = 100
    T_step = 0.01
    n_steps = 1000

    # 1. Prepare
    psi_init, phi_post = get_lchs_states(beta, r_target, r_prime, N_dim)

    # 2. Run
    times, u_vals, v_vals, omega_eff = run_simulation(omega, gamma_damp, T_step, n_steps, N_dim, psi_init, phi_post)
    # print(times)
    # print(u_vals)
    # u_vals = np.exp(4*times) * np.array(u_vals)
    # v_vals = np.exp(4*times) * np.array(v_vals)

    # 3. Classical Solution (Damped Harmonic Oscillator)
    # u'' + 2*gamma_damp*u' + omega^2*u = 0, with u(0)=1, u'(0)=0
    omega_d = np.sqrt(max(omega**2 - gamma_damp**2, 0.0))
    if omega_d <= 0.0:
        raise ValueError("gamma_damp must be < omega for underdamped solution")
    exact_u = np.exp(-gamma_damp * times) * (
        np.cos(omega_d * times) + (gamma_damp / omega_d) * np.sin(omega_d * times)
    )
    exact_v = -(omega**2 / omega_d) * np.exp(-gamma_damp * times) * np.sin(omega_d * times)

    # 4. Visualization

    # The quantum state effectively sees this frequency because <K> = 0
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)

    # Position Plot
    plt.plot(times, u_vals, 'b.-', label='LCHS (Qubit |0>)', linewidth=2)
    plt.plot(times, exact_u, 'k--', label='Classical damped u(t)', alpha=0.6)
    plt.title("Position $u(t)$")
    plt.xlabel("Time")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Velocity Plot
    plt.subplot(2, 1, 2)
    plt.plot(times, v_vals, 'r.-', label='LCHS (Qubit |1>)', linewidth=2)

    plt.plot(times, exact_v, 'k--', label='Classical damped v(t)', alpha=0.6)

    plt.title("Velocity (Phase Check)")
    plt.xlabel("Time")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [147]:
if __name__ == "__main__":
    main()

1. Preparing States (N=100, r_target=8)...
   States ready.
2. Running Simulation...
(5.423397116187087e-18-0.9105320566327622j)
[0.09105321-3.42441401j 0.09105321+3.42441401j]
   omega_eff=3.424


NameError: name 'L_shift' is not defined

In [ ]:
L = np.array([[0, 4], [4, 0]]) + np.array([[4, 0], [0, 4]])
np.linalg.eigvals(L)

array([8., 0.])